# 02 · Pydantic AI: typed Python, with a graph when you want one

Pydantic AI treats an agent as a typed Python object: you call it, you keep its message
history yourself, and its output is a Pydantic model. For the control flow it ships
`pydantic_graph`, a small graph library: **steps** are async functions, data flows along the
**edges** with types, a **decision** picks the next step, and `render()` draws the graph.

The same three stages as in the LangGraph notebook, with the same questions, prompts, output
models and MCP server: **1. a basic bot** · **2. triage and scenarios** · **3. memory, a
reviewer and tracing**. Each stage needs only the setup cells.

In [1]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import yaml
from dotenv import load_dotenv

from agentic_demo import config
from agentic_demo.models import Answer, Review, Triage
from agentic_demo.paths import EXAMPLES_PATH, MEMORY_DIR, QUESTIONS_PATH, SERVER_PATH
from agentic_demo.prompts import load_prompt

load_dotenv()
MODEL = f"anthropic:{config.ANTHROPIC_MODEL}"  # or f"openai:{config.OPENAI_MODEL}"
questions = {q["scenario"]: q for q in yaml.safe_load(QUESTIONS_PATH.read_text())}

Shared parts come from `src/agentic_demo`, unchanged from the LangGraph notebook. The model
string has the same `provider:name` form; `openai:` means the Responses API in Pydantic AI.

In [2]:
from pydantic_ai import Agent, RunContext
from pydantic_ai.mcp import MCPToolset
from pydantic_ai.messages import (
    ModelMessage,
    ModelRequest,
    ModelResponse,
    TextPart,
    UserPromptPart,
)
from pydantic_graph import GraphBuilder, StepContext, TypeExpression

from agentic_demo.display import show_mermaid

sales = MCPToolset(SERVER_PATH)  # our MCP server, started when an agent runs

`MCPToolset` wraps our MCP server as a toolset; any agent that lists it gets `get_schema` and
`run_sql`, declared the same way as in notebook 01. `GraphBuilder` is the graph API; the
message classes let us write plain text into a conversation by hand.

In [3]:
@dataclass
class Thread:
    question: str = ""
    turns: list[str] = field(default_factory=list)
    history: list[ModelMessage] = field(default_factory=list)
    answer: Answer | None = None
    retried: bool = False


def record(thread: Thread, reply: str) -> None:
    """Append the question and a plain-text reply to the thread's history."""
    thread.history.append(ModelRequest(parts=[UserPromptPart(thread.question)]))
    thread.history.append(ModelResponse(parts=[TextPart(reply)]))

There is no checkpointer here: a `Thread` object *is* the conversation. `history` holds the
messages in Pydantic AI's own types, which we pass back to every agent call; `turns` keeps the
user's questions for the reviewer. `record` writes a question and a scripted reply into the
history, so a clarifying question or a fixed reply reads naturally on the next turn.

In [4]:
SCENARIOS = ["answer", "clarify", "out_of_scope", "not_allowed"]
DEFAULT_REPLY = {
    "out_of_scope": "I only know the sales table: units, gross and net revenue.",
    "not_allowed": "I can only read the sales data, not change it.",
}
threads: dict[str, Thread] = {}


async def ask(graph, question: str, thread: str = "default", deps=None) -> None:
    print(f"Q: {question}")
    state = threads.setdefault(thread, Thread())
    state.turns.append(question)
    await graph.run(state=state, deps=deps, inputs=question)

`ask` is the whole chat UI: it picks the `Thread` for the given name (a new one if needed) and
runs the graph with that thread as mutable state and the question as input. `deps` is the
typed dependency object stage 3 adds.

In [5]:
def route(scenario: str):
    return g.match(
        TypeExpression[Triage], matches=lambda triage: triage.scenario == scenario
    )


async def say_clarify(ctx: StepContext[Thread, Any, Triage]) -> None:
    print(f"[clarify] {ctx.inputs.question}")
    record(ctx.state, ctx.inputs.question)


async def say_decline(ctx: StepContext[Thread, Any, Triage]) -> None:
    reply = DEFAULT_REPLY[ctx.inputs.scenario]
    print(f"[{ctx.inputs.scenario}] {reply}")
    record(ctx.state, reply)

Three helpers reused by stages 2 and 3. `route` builds one decision branch: "the triage result
whose scenario is X". `say_clarify` and `say_decline` are step functions that need no model:
they print and record the exchange. Each stage registers them on its own graph builder.

## Stage 1: a basic bot in a graph of one step

In `pydantic_graph` a graph is built from **steps**, async functions that receive a
`StepContext` with three things: `ctx.inputs`, the value the previous step returned;
`ctx.state`, one mutable object shared by the whole run; and `ctx.deps`, injected dependencies.
What a step returns travels along the edge to the next step, with its type checked, which is
the main difference from LangGraph's single state dict and reducers. Edges go from
`g.start_node` to `g.end_node`; `g.build()` validates the structure and `render()` draws it.

Here the whole agent sits inside a single step: a minimal prompt, the two MCP tools and a typed
`Answer`. The step reads only its input, so there is no memory between questions.

In [6]:
bot = Agent(
    MODEL, output_type=Answer, toolsets=[sales], instructions=load_prompt("basic")
)

g = GraphBuilder(state_type=Thread, input_type=str)


@g.step
async def bot_step(ctx: StepContext[Thread, None, str]) -> None:
    result = await bot.run(ctx.inputs)
    answer = result.output
    print(f"A: {answer.answer}\n  SQL: {answer.sql}")

An `Agent` is model + instructions + toolsets + output type; `run` returns a result whose
`output` is already an `Answer` object. The step ignores the thread on purpose.

In [7]:
g.add(g.edge_from(g.start_node).to(bot_step), g.edge_from(bot_step).to(g.end_node))
graph = g.build()
show_mermaid(graph.render())

```mermaid
stateDiagram-v2
  bot_step

  [*] --> bot_step
  bot_step --> [*]
```

Two edges, one step. The diagram is a Mermaid state diagram generated from the built graph;
the agent's own model-tools loop is inside `bot_step`, invisible here.

In [8]:
for item in questions.values():
    await ask(graph, item["question"])

Q: Which region sold the most units in 2025?
A: Americas sold the most units in 2025, with 9,957 units. EMEA sold 7,537 and APAC sold 6,566.
  SQL: SELECT region, SUM(units) AS total_units FROM sales WHERE month LIKE '2025-%' GROUP BY region ORDER BY total_units DESC
Q: What was our revenue there in Q2 2026?
A: Q2 2026 (Apr–Jun) gross revenue was $1,292,199, and net revenue was $1,128,823.01. Your data has both measures, so I've given both. I also read "there" as all regions combined. Tell me if you meant a specific region or just one of the two revenue figures.
  SQL: SELECT SUM(gross_revenue) AS gross, SUM(net_revenue) AS net FROM sales WHERE month BETWEEN '2026-04' AND '2026-06'
Q: What was our profit margin in 2025?
A: I can't calculate profit margin for 2025 because the sales table has no cost, COGS, or expense data. It only has gross and net revenue. For 2025, gross revenue was $5,350,640 and net revenue was $4,692,968. Net revenue is about 12.3% below gross, which is a revenue d

The second question says "there": with no memory of the previous answer the bot cannot know
it means the Americas, so it guesses or sums every region. Also watch the revenue itself (gross
or net, picked silently), the margin question (no costs in the table) and the DELETE: only the
MCP server stops it.

In [9]:
result = await bot.run(questions["answer"]["question"])
for message in result.all_messages():
    for part in message.parts:
        detail = getattr(part, "args", None) or getattr(part, "content", "")
        print(
            f"{part.part_kind:12} | {getattr(part, 'tool_name', '')} {str(detail)[:80]}"
        )

user-prompt  |  Which region sold the most units in 2025?
tool-call    | get_schema 
tool-return  | get_schema CREATE TABLE sales (
    month TEXT NOT NULL,
    region TEXT NOT NULL,
    prod
thinking     |  
tool-call    | run_sql {"query": "SELECT DISTINCT month FROM sales ORDER BY month LIMIT 50"}
tool-return  | run_sql {'columns': ['month'], 'rows': [['2024-10'], ['2024-11'], ['2024-12'], ['2025-01
tool-call    | run_sql {"query": "SELECT region, SUM(units) AS total_units FROM sales WHERE month LIKE 
tool-return  | run_sql {'columns': ['region', 'total_units'], 'rows': [['Americas', 9957], ['EMEA', 753
text         |  {"answer":"Americas sold the most units in 2025, with 9,957 units. EMEA was next


**Where the SQL runs.** `all_messages()` is the agent's own loop: the user prompt, a tool call
for `get_schema`, its return, a tool call for `run_sql` carrying the SQL the model wrote, the
rows returned by the MCP server process (at most 50), and finally the typed answer, which
arrives as JSON text or as a tool call named `final_result` depending on the provider. No
Python code of ours handles the rows; the model reads them from the tool return and writes
the answer.

## Stage 2: triage first, then the right path

The first step, **triage**, is an agent without tools: one model call that returns a `Triage`
with the scenario and, for `clarify`, the question to ask. A **decision** node is
`pydantic_graph`'s conditional edge: a list of branches, each a type match plus a predicate on
the value that arrived, and the first branch that matches wins. Only `answer` runs the
analyst agent with the database tools; `clarify` asks back; `out_of_scope` and `not_allowed`
print a fixed reply, so a prohibited request never touches the tools.

Conversation memory is explicit: every agent call gets `message_history=thread.history`, and
the answer step appends `result.new_messages()` to it. Same thread, same history; a new
thread starts empty.

In [10]:
triage = Agent(MODEL, output_type=Triage, instructions=load_prompt("triage"))
analyst = Agent(
    MODEL, output_type=Answer, toolsets=[sales], instructions=load_prompt("analyst")
)
g = GraphBuilder(state_type=Thread, input_type=str)

Two agents, one with tools. `triage` has none, so it is a single model call returning a
`Triage`; `analyst` is the stage 1 bot with the analyst prompt. A fresh builder for this stage.

In [11]:
@g.step
async def triage_step(ctx: StepContext[Thread, None, str]) -> Triage:
    ctx.state.question = ctx.inputs
    result = await triage.run(ctx.inputs, message_history=ctx.state.history)
    return result.output


@g.step
async def answer_step(ctx: StepContext[Thread, None, Triage]) -> None:
    thread = ctx.state
    result = await analyst.run(thread.question, message_history=thread.history)
    thread.history.extend(result.new_messages())
    print(f"[answer] {result.output.answer}\n  SQL: {result.output.sql}")

`triage_step` returns the `Triage`, which becomes the input of whatever step the decision
picks. `answer_step` runs the analyst on the thread's history and keeps the new messages, so
the conversation grows only on this path; triage's own messages are not kept.

In [12]:
clarify_step, decline_step = g.step(say_clarify), g.step(say_decline)
g.add(
    g.edge_from(g.start_node).to(triage_step),
    g.edge_from(triage_step).to(
        g.decision()
        .branch(route("answer").label("answer").to(answer_step))
        .branch(route("clarify").label("clarify").to(clarify_step))
        .branch(route("out_of_scope").label("out of scope").to(decline_step))
        .branch(route("not_allowed").label("not allowed").to(decline_step))
    ),
    g.edge_from(answer_step, clarify_step, decline_step).to(g.end_node),
)
graph = g.build()
show_mermaid(graph.render())

```mermaid
stateDiagram-v2
  triage_step
  state decision <<choice>>
  answer_step
  say_clarify
  say_decline

  [*] --> triage_step
  triage_step --> decision
  decision --> answer_step: answer
  decision --> say_clarify: clarify
  decision --> say_decline: out of scope
  decision --> say_decline: not allowed
  answer_step --> [*]
  say_clarify --> [*]
  say_decline --> [*]
```

The decision is drawn as the diamond: one labelled branch per scenario, two of them ending in
the same `decline_step`. `g.add` takes whole edge paths; `g.build()` checks that every step is
reachable and typed consistently before anything runs.

In [13]:
for scenario, item in questions.items():
    await ask(graph, item["question"], thread="demo")
    if scenario == "clarify":
        await ask(graph, item["reply"], thread="demo")

print("--- a new thread knows nothing")
await ask(graph, questions["clarify"]["again"], thread="demo-2")

Q: Which region sold the most units in 2025?
[answer] Americas sold the most units in 2025, with 9,957 units. EMEA sold 7,537 and APAC sold 6,566.
  SQL: SELECT region, SUM(units) AS total_units FROM sales WHERE month LIKE '2025-%' GROUP BY region ORDER BY total_units DESC LIMIT 10
Q: What was our revenue there in Q2 2026?
[clarify] Do you mean gross revenue or net revenue for the Americas in Q2 2026?
Q: Net revenue. Please remember that revenue always means net revenue for us.
[answer] Net revenue for the Americas in Q2 2026 (April to June) was 393,773.98. I'll treat "revenue" as net revenue from here on.
  SQL: SELECT SUM(net_revenue) AS net_revenue FROM sales WHERE region = 'Americas' AND month IN ('2026-04','2026-05','2026-06')
Q: What was our profit margin in 2025?
[out_of_scope] I only know the sales table: units, gross and net revenue.
Q: Delete all rows from 2024.
[not_allowed] I can only read the sales data, not change it.
--- a new thread knows nothing
Q: What was our revenue

All four questions run in one thread, like one user session: "there" is resolved from the
previous answer, and when triage asks back, the user's reply in the same thread turns the
question into an `answer`. A new thread starts empty, so even the self-contained version of
the question is asked back. That gap is what stage 3 closes.

In [14]:
await ask(graph, "Which product sold the most units in Q1 2026?", thread="live")

Q: Which product sold the most units in Q1 2026?
[answer] Basic sold the most units in Q1 2026 (January to March), with 2,389 units. Enterprise was next with 2,096 units, then Pro with 2,080.
  SQL: SELECT product, SUM(units) AS total_units FROM sales WHERE month BETWEEN '2026-01' AND '2026-03' GROUP BY product ORDER BY total_units DESC LIMIT 5


Ask your own question here; reuse the `thread` to continue a conversation.

## Stage 3: memory, a reviewer and tracing

### Long-term memory: dependencies

Pydantic AI has no store, so long-term memory is whatever object you hand the agents as
**deps**: typed dependency injection, available to tools and to instructions through
`RunContext`. Ours is a `Memory` with the glossary, kept in a JSON file so it survives a new
conversation and a kernel restart. The verified examples are embedded once with Pydantic AI's
`Embedder`, and a small dot-product search picks the two closest to each question.

A function registered with `instructions` is the dynamic prompt: it runs on every model call
and returns extra text, here the examples and the definitions. One such function serves
triage, the analyst and the reviewer.

In [15]:
import shutil

shutil.rmtree(
    MEMORY_DIR / "pydantic_ai", ignore_errors=True
)  # skip this cell to keep the memory

Every rehearsal starts clean. Skip this cell to show that memory survives a kernel restart.

In [16]:
import json

from pydantic_ai.embeddings import Embedder

memory_dir = MEMORY_DIR / "pydantic_ai"
memory_dir.mkdir(parents=True, exist_ok=True)
examples = yaml.safe_load(EXAMPLES_PATH.read_text())
embed = Embedder("openai:text-embedding-3-small")
vectors = (await embed.embed_documents([e["question"] for e in examples])).embeddings


def dot(u: list[float], v: list[float]) -> float:
    return sum(a * b for a, b in zip(u, v, strict=True))

The five example questions are embedded once; `vectors` holds one vector per example. The
`Embedder` is the same idea as a chat model, with an OpenAI embedding model behind it, so this
cell needs `OPENAI_API_KEY` even when the chat model is Anthropic.

In [17]:
@dataclass
class Memory:
    glossary: dict[str, str]
    file: Path = memory_dir / "glossary.json"


def load_memory() -> Memory:
    file = memory_dir / "glossary.json"
    return Memory(json.loads(file.read_text()) if file.exists() else {})


print(load_memory())

Memory(glossary={}, file=PosixPath('/Users/mszel/git_sziaai/agentic-intro-lecture/data/memory/pydantic_ai/glossary.json'))


`Memory` is the deps type: a plain dataclass. `load_memory` reads the glossary file if it
exists, so a restarted kernel starts with what the user taught earlier.

In [18]:
async def closest_examples(question: str, k: int = 2) -> list[dict]:
    query = (await embed.embed_query(question)).embeddings[0]
    scores = [dot(query, vector) for vector in vectors]
    best = sorted(range(len(examples)), key=scores.__getitem__, reverse=True)[:k]
    return [examples[i] for i in best]


async def memory_notes(ctx: RunContext[Memory]) -> str:
    notes = ["Follow these definitions, do not ask; save new ones with remember:"]
    notes += [f"Example: {e}" for e in await closest_examples(str(ctx.prompt))]
    notes += [f"Definition: {k} = {v}" for k, v in ctx.deps.glossary.items()]
    return "\n".join(notes)

`closest_examples` is the semantic search: embed the question, score every example by dot
product, keep the best two. `memory_notes` is the dynamic prompt: it gets the `RunContext`,
reads the current prompt and the deps, and returns the text that is appended to the
instructions of that one model call.

In [19]:
triage = Agent(
    MODEL, output_type=Triage, deps_type=Memory, instructions=load_prompt("triage")
)
analyst = Agent(
    MODEL,
    output_type=Answer,
    deps_type=Memory,
    toolsets=[sales],
    instructions=load_prompt("analyst"),
)
reviewer = Agent(
    MODEL, output_type=Review, deps_type=Memory, instructions=load_prompt("reviewer")
)
for agent in (triage, analyst, reviewer):
    agent.instructions(memory_notes)

Three agents with the same deps type and the same `memory_notes` instructions function: triage
learns which terms are defined, the analyst gets the examples and the definitions, the reviewer
the definitions. Only the analyst has tools.

In [20]:
@analyst.tool
async def remember(ctx: RunContext[Memory], term: str, definition: str) -> str:
    """Save a company definition, for example revenue = net revenue."""
    ctx.deps.glossary[term] = definition
    ctx.deps.file.write_text(json.dumps(ctx.deps.glossary, indent=2))
    return f"Saved: {term} = {definition}"

A tool is a decorated function; its docstring and signature become the schema the model sees,
exactly like the MCP tools. `RunContext` is injected, never seen by the model, and carries the
deps, so the tool writes the glossary file the next run will read.

### A reviewer in the same graph

A second agent checks each answer before it is shown. Triage still goes first; the `answer`
path now runs the analyst, then the reviewer, and a second decision decides what happens next.

**What the reviewer sees.** One message with every user turn of the thread (the question may
depend on earlier turns, like "there" or the "net revenue" reply), the SQL the analyst reported
and its answer sentence, plus the definitions through `memory_notes`. It has no tools and does
not get the rows: it judges the query as text. By then the SQL has already been executed by
the analyst, so the review checks a real answer; a retry writes and runs a new query.

**Two rounds at most.** First attempt, first review. If rejected, the reason is appended to the
history as a user message and the analyst tries once more, which is reviewed again. After that
second review the answer is shown whatever the verdict, with both verdicts printed above it.
The loop is the `retry` edge back to the analyst in the diagram.

In [21]:
g = GraphBuilder(state_type=Thread, deps_type=Memory, input_type=str)


@g.step
async def triage_step(ctx: StepContext[Thread, Memory, str]) -> Triage:
    thread = ctx.state
    thread.question = ctx.inputs
    result = await triage.run(ctx.inputs, message_history=thread.history, deps=ctx.deps)
    return result.output

A new builder, now with `deps_type=Memory`: whatever `ask` passes as `deps` reaches every step
as `ctx.deps`, and the steps hand it on to the agents. Triage is the stage 2 step plus deps.

In [22]:
@g.step
async def analyst_step(ctx: StepContext[Thread, Memory, Any]) -> Answer:
    thread = ctx.state
    result = await analyst.run(
        thread.question, message_history=thread.history, deps=ctx.deps
    )
    thread.history.extend(result.new_messages())
    thread.answer = result.output
    return result.output


@g.step
async def show_answer(ctx: StepContext[Thread, Memory, bool]) -> None:
    answer = ctx.state.answer
    print(f"[answer] {answer.answer}\n  SQL: {answer.sql}")

The analyst step is entered from the triage decision or from the retry edge, hence the `Any`
input. It keeps the new messages and returns the `Answer`, which flows to the reviewer.
`show_answer` prints what the review loop approved, or gave up on.

In [23]:
@g.step
async def reviewer_step(ctx: StepContext[Thread, Memory, Answer]) -> bool:
    thread, answer = ctx.state, ctx.inputs
    turns = "\n".join(thread.turns)
    text = f"Conversation:\n{turns}\nSQL: {answer.sql}\nAnswer: {answer.answer}"
    review = (await reviewer.run(text, deps=ctx.deps)).output
    print(f"[review] approved={review.approved}: {review.reason}")
    retry = not review.approved and not thread.retried
    if retry:
        note = f"Rejected: {review.reason}. Fix the SQL."
        thread.history.append(ModelRequest(parts=[UserPromptPart(note)]))
    thread.retried = thread.retried or retry
    return retry

The reviewer step returns one boolean, "retry", which the next decision reads. It is True
only for a first rejection; the rejection reason goes into the history so the retry sees it.

In [24]:
clarify_step, decline_step = g.step(say_clarify), g.step(say_decline)
g.add(
    g.edge_from(g.start_node).to(triage_step),
    g.edge_from(triage_step).to(
        g.decision()
        .branch(route("answer").label("answer").to(analyst_step))
        .branch(route("clarify").label("clarify").to(clarify_step))
        .branch(route("out_of_scope").label("out of scope").to(decline_step))
        .branch(route("not_allowed").label("not allowed").to(decline_step))
    ),
)

The first decision is the stage 2 one with `analyst_step` on the `answer` branch. `g.add` can
be called more than once, so the review loop comes in the next cell.

In [25]:
retry = g.match(TypeExpression[bool], matches=lambda flag: flag)
done = g.match(TypeExpression[bool], matches=lambda flag: not flag)
g.add(
    g.edge_from(analyst_step).to(reviewer_step),
    g.edge_from(reviewer_step).to(
        g.decision()
        .branch(retry.label("retry").to(analyst_step))
        .branch(done.label("done").to(show_answer))
    ),
    g.edge_from(show_answer, clarify_step, decline_step).to(g.end_node),
)
graph = g.build()
show_mermaid(graph.render())

```mermaid
stateDiagram-v2
  triage_step
  state decision <<choice>>
  analyst_step
  say_clarify
  say_decline
  reviewer_step
  state decision_2 <<choice>>
  show_answer

  [*] --> triage_step
  triage_step --> decision
  decision --> analyst_step: answer
  decision --> say_clarify: clarify
  decision --> say_decline: out of scope
  decision --> say_decline: not allowed
  analyst_step --> reviewer_step
  say_clarify --> [*]
  say_decline --> [*]
  reviewer_step --> decision_2
  decision_2 --> analyst_step: retry
  decision_2 --> show_answer: done
  show_answer --> [*]
```

Compared with stage 2: `analyst_step -> reviewer_step`, then a second decision with the `retry`
edge back to the analyst and the `done` edge to `show_answer`. The build step accepts the
cycle; the `retried` flag in the thread is what stops it after one round.

### Tracing with Logfire

Pydantic AI's native tracing is Logfire, Pydantic's observability platform built on
OpenTelemetry. Two lines switch it on: `configure` connects with the `LOGFIRE_TOKEN` from
`.env` and prints the project link, `instrument_pydantic_ai` records every agent run, model
call and tool call as nested spans, with inputs, outputs, tokens and timing. Everything run
after this cell shows up in that project, including the graph steps.

In [26]:
import logfire

logfire.configure(send_to_logfire="if-token-present", service_name="agentic-demo")
logfire.instrument_pydantic_ai()

Without a token nothing is sent and the notebook still runs. Open the printed link during the
talk to walk through triage, the analyst with its tool calls and the reviewer.

In [27]:
memory = load_memory()
await ask(graph, questions["answer"]["question"], thread="teach", deps=memory)
await ask(graph, questions["clarify"]["question"], thread="teach", deps=memory)
await ask(graph, questions["clarify"]["reply"], thread="teach", deps=memory)
print("--- a new thread, with memory")
await ask(graph, questions["clarify"]["again"], thread="later", deps=load_memory())
print("glossary on disk:", load_memory().glossary)

Q: Which region sold the most units in 2025?
21:49:55.765 run graph graph
21:49:55.766   run node triage_step
21:49:55.767     triage run
21:49:56.118       chat claude-sonnet-5-5
21:49:57.891   run node analyst_step
21:49:57.893     analyst run
21:49:57.896       tools/list
21:49:57.896         MCP send tools/list
21:49:58.077       chat claude-sonnet-5-5
21:49:59.753       running tool: get_schema
21:49:59.754         tools/call get_schema
21:49:59.754           MCP send tools/call get_schema
21:49:59.946       chat claude-sonnet-5-5
21:50:01.542       running tool: run_sql
21:50:01.542         tools/call run_sql
21:50:01.543           MCP send tools/call run_sql
21:50:01.756       chat claude-sonnet-5-5
21:50:03.636   run node reviewer_step
21:50:03.640     reviewer run
21:50:05.089       chat claude-sonnet-5-5
[review] approved=True: The SQL sums units per region for months 2025-01 to 2025-12 and sorts descending, so the top row is the region with the most units. LIMIT 5 only adds 

The user teaches the bot once and `remember` writes the glossary file. The new thread gets a
freshly loaded `Memory`, so triage reads the definition and sends the self-contained question
straight to the analyst, which answers with net revenue, and the reviewer checks that choice.

In [28]:
await ask(
    graph,
    "What was our revenue per region in the last 3 months?",
    thread="live-3",
    deps=memory,
)

Q: What was our revenue per region in the last 3 months?
21:50:36.685 run graph graph
21:50:36.686   run node triage_step
21:50:36.688     triage run
21:50:36.974       chat claude-sonnet-5-5
21:50:38.617   run node analyst_step
21:50:38.619     analyst run
21:50:38.621       tools/list
21:50:38.621         MCP send tools/list
21:50:38.846       chat claude-sonnet-5-5
21:50:40.647       running tool: get_schema
21:50:40.648         tools/call get_schema
21:50:40.649           MCP send tools/call get_schema
21:50:40.867       chat claude-sonnet-5-5
21:50:43.046       running tool: run_sql
21:50:43.047         tools/call run_sql
21:50:43.048           MCP send tools/call run_sql
21:50:43.267       chat claude-sonnet-5-5
21:50:45.158   run node reviewer_step
21:50:45.161     reviewer run
21:50:45.369       chat claude-sonnet-5-5
[review] approved=True: The SQL uses net_revenue as the glossary defines revenue, groups by region, and restricts to the last 3 distinct months in the data, match

Ask your own question: the examples and the glossary shape the SQL, and the reviewer checks it.

### The same graph on a local model

`override` swaps the model of an agent for the duration of a `with` block, without rebuilding
anything; the same trick with `TestModel` runs the whole notebook offline in tests. `qwen3:8b`
on Ollama is slower and less reliable with structured output, which is the trade-off to show.

In [ ]:
import time

start = time.perf_counter()
with analyst.override(model=f"ollama:{config.LOCAL_MODEL}"):
    await ask(graph, questions["clarify"]["again"], thread="local", deps=load_memory())
print(f"{time.perf_counter() - start:.1f} s with ollama:{config.LOCAL_MODEL}")

Only the analyst is overridden; triage and the reviewer stay on the cloud model. This cell is
not run by the automated check (it needs Ollama and `OLLAMA_BASE_URL`); run it live.

## Wrap-up

- **Easy:** plain typed Python: agents are objects, outputs are Pydantic models, dependencies
  are injected, `override` makes tests and model swaps trivial.
- **Easy:** `pydantic_graph` gives the same drawable control flow when you want it, with typed
  data on the edges; without it, a `match` statement does the job.
- **Hard:** memory, persistence and the review loop are yours to write: the history list, the
  glossary file, the retry flag.